# Wrangle SDO78 Data Files

This notebook loads all Excel files from the ~/local-share/raw/sdo78 folder, containing the '100 dagen monitor' questionnaire data for the years 2021, 2022, and 2023, and merges into one useable dataframe.
Accompanying keys files are also loaded, as well as a preselection of questions that we want to include in the final dataset, which is stored in the df_q_selection dataframe.

The goal is to create a final dataset that will contain the following columns:
- SINH_ID
- Year (2021, 2022, 2023)
- All questions that were preselected in the df_q_selection dataframe.

## Import libraries and load data

In [ ]:
import pandas as pd
from pathlib import Path
import os

In [ ]:
# Define the data directory using the current user's home directory
current_user = os.getlogin()
data_dir = Path(f"/home/{current_user}/local-share/raw/sdo78")
print(f"Data directory: {data_dir}")

In [ ]:
# Load each Excel file into a separate dataframe
df_2021 = pd.read_excel(data_dir / '100DAGEN_VT_HU_2021.xlsx')
df_2021_keys = pd.read_excel(data_dir / '2021_keys.xlsx')
df_2022 = pd.read_excel(data_dir / '2022_2023_HU_100D_VT_codeboek.xlsx')
df_2022_keys = pd.read_excel(data_dir / '2022_keys.xlsx')
df_2023 = pd.read_excel(data_dir / '2023_2024_100dagen_HU_VT_BASE_codeboek.xlsx')
df_2023_keys = pd.read_excel(data_dir / '2023_keys.xlsx')
df_q_selection = pd.read_excel(data_dir / 'Q_selection.xlsx')

print("All Excel files loaded successfully!")
print(f"\nDataframes created:")
print(f"- df_2021: {df_2021.shape}")
print(f"- df_2021_keys: {df_2021_keys.shape}")
print(f"- df_2022: {df_2022.shape}")
print(f"- df_2022_keys: {df_2022_keys.shape}")
print(f"- df_2023: {df_2023.shape}")
print(f"- df_2023_keys: {df_2023_keys.shape}")
print(f"- df_q_selection: {df_q_selection.shape}")

## Merge data files with keys files
To couple this data with other drop-out data, we need the SINH_ID. 
- For both 2021 and 2022 student numbers are directly available of which we can derive the SINH_ID.
- For 2023 SINH_ID is available in the keys file, which we can couple on resp_id. 

So only for 2021 and 2023 we need to merge the data files with the keys files.

In [ ]:
# Rename join columns in keys dataframes to match the main dataframes
df_2021_keys.rename(columns={'Benadernummer': 'resp_nr'}, inplace=True)
df_2023_keys.rename(columns={'Benadernummer2': 'resp_nr2'}, inplace=True)

In [ ]:
# Merge the dataframes based on the keys
merged_2021 = pd.merge(df_2021, df_2021_keys, on='resp_nr', how='inner')
merged_2023 = pd.merge(df_2023, df_2023_keys, on='resp_nr2', how='inner')

merged_2022 = df_2022.copy()  # No merge needed for 2022 as SINH_ID is directly available

## Data verification

In [ ]:
# Add year column to each merged dataframe
merged_2021['year'] = 2021
merged_2022['year'] = 2022
merged_2023['year'] = 2023

In [ ]:
# Verify if all questions in df_q_selection
# are present in the merged dataframes as columns
missing_questions_2021 = set(df_q_selection['Q_ID']) - set(merged_2021.columns)
missing_questions_2022 = set(df_q_selection['Q_ID']) - set(merged_2022.columns)
missing_questions_2023 = set(df_q_selection['Q_ID']) - set(merged_2023.columns)

Question HU22_01 does not appear in the 2021 questionnaire data, so it must be excluded. 

In [ ]:
# Exclude question HU22_01 from the questions
# Delete row with SINH_ID = 'HU22_01' from df_q_selection
df_q_selection = df_q_selection[df_q_selection['Q_ID'] != 'HU22_01']

del missing_questions_2021, missing_questions_2022, missing_questions_2023  

## Find SINH_ID
For each record we need to find the corresponding SINH_ID. We use the following approach:
1. Load merged student drop-out data for 2021-2023
2. Look-up the SINH_ID based on student number (2021 and 2022), combined with CROHO and year. 

Please note: this is a cutting-many-corners approach, which is not very efficient or clean, but it works for our purposes (seeing if this data is feasible to add in the model).

In [ ]:
# Load student drop-out data for 2021-2023
# Note: current_user already defined at the beginning of notebook
DATA_DIR = Path(f"/home/{current_user}/local-share/processed")
combined_path = DATA_DIR / "v2_combined.csv"

dropout_data = pd.read_csv(combined_path, sep=None, engine="python", encoding="utf-8-sig")

In [ ]:
# Try left join of final_df on SINH_ID
enriched_2023 = pd.merge(merged_2023, dropout_data, left_on='Sinh_id', right_on='SINH_ID', how='left')

# Check how many 2023 records have a match in the dropout data
matched_2023 = enriched_2023[enriched_2023['year'] == 2023]['SINH_ID'].notna().sum()
total_2023 = enriched_2023[enriched_2023['year'] == 2023].shape[0]
print(f"Matched records for 2023: {matched_2023} out of {total_2023}")

For 2023 the unmatched records concern Ad students, which were out of scope for the drop-out data collection. 

In [ ]:
# Try left join on student number (2022)
enriched_2022 = pd.merge(merged_2022, dropout_data, left_on=['studentnr', 'CROHO', 'STARTJAAR_OPLEIDING'], 
                     right_on=['sdo1_characteristics_student_number', 'sdo5_degree_degree_code', 'sdo5_degree_COLLEGEJAAR'], how='left')

# Check how many 2022 records have a match in the dropout data
matched_2022 = enriched_2022[enriched_2022['year'] == 2022]['studentnr'].notna().sum()
total_2022 = enriched_2022[enriched_2022['year'] == 2022].shape[0]
print(f"Matched records for 2022: {matched_2022} out of {total_2022}")

In [ ]:
# Try left join on student number (2021)
enriched_2021 = pd.merge(merged_2021, dropout_data, left_on=['STUDENTNUMMER', 'CROHO_x', 'STARTJAAR_OPLEIDING_x'], 
                     right_on=['sdo1_characteristics_student_number', 'sdo5_degree_degree_code', 'sdo5_degree_COLLEGEJAAR'], how='left')

# Check how many 2021 records have a match in the dropout data
matched_2021 = enriched_2021[enriched_2021['year'] == 2021]['STUDENTNUMMER'].notna().sum()
total_2021 = enriched_2021[enriched_2021['year'] == 2021].shape[0]
print(f"Matched records for 2021: {matched_2021} out of {total_2021}")

## Union all years
Add all three years together, and keep only the preselected questions.

In [ ]:
# Union the three merged dataframes into a single dataframe
combined_df = pd.concat([enriched_2021, enriched_2022, enriched_2023], ignore_index=True)

# Keep only the columns that are in df_q_selection, year and SINH_ID
q_columns = [col for col in df_q_selection['Q_ID'] if col in combined_df.columns]
final_df = combined_df[['SINH_ID', 'year'] + q_columns].copy()

# Create a mapping of Q_ID to Topic to rename columns with both Q_ID and Topic
q_to_topic = dict(zip(df_q_selection['Q_ID'], df_q_selection['Topic']))
rename_dict = {q_id: f"{q_id} ({q_to_topic[q_id]})" for q_id in q_columns if q_id in q_to_topic}

# Rename the Q_ID columns to include Topic information
final_df.rename(columns=rename_dict, inplace=True)

Some of the records do not have a SINH_ID, and for now we did not investigate why. For now we drop these, but this should be investigated further if we want to use this data in the model.

In [ ]:
# Drop all records where SINH_ID is missing (i.e., no match in dropout data)
final_df = final_df[final_df['SINH_ID'].notna()]

### Data Quality Summary

This cell reports SINH_ID linkage success rates per year and overall record retention, documenting data quality before further processing.

In [ ]:
# Data quality report: SINH_ID linkage success per year
print("=== SDO78 Data Quality Report ===")
print()

# Match rates per year (from the individual year merges above)
total_21 = enriched_2021.shape[0]
matched_21 = enriched_2021['SINH_ID'].notna().sum()

total_22 = enriched_2022.shape[0]
matched_22 = enriched_2022['SINH_ID'].notna().sum()

total_23 = enriched_2023[enriched_2023['year'] == 2023].shape[0]
matched_23 = enriched_2023[enriched_2023['year'] == 2023]['SINH_ID'].notna().sum()

print(f"{'Year':<6} {'SDO78 records':<18} {'Matched to dropout data':<25} {'Match rate'}")
print("-" * 65)
for year, total, matched in [(2021, total_21, matched_21), (2022, total_22, matched_22), (2023, total_23, matched_23)]:
    rate = matched / total * 100 if total > 0 else 0
    print(f"{year:<6} {total:<18} {matched:<25} {rate:.1f}%")

print()
records_before = combined_df.shape[0]
records_after = final_df.shape[0]
dropped = records_before - records_after
print(f"Records before dropping unmatched: {records_before:,}")
print(f"Records after  dropping unmatched: {records_after:,}")
print(f"Dropped (no SINH_ID match): {dropped:,} ({dropped/records_before*100:.1f}%)")
print()
print("Note: Unmatched 2023 records are primarily Ad students (out of scope).")
print("      Unmatched 2021/2022 records have not been fully investigated; flagged for future review.")

### Some data cleaning

In [ ]:
# Set all -1 answers to NaN (i.e., unanswered)
final_df.replace(-1, pd.NA, inplace=True)

In [ ]:
# Count per record how many questions have missing (i.e., not answered)
final_df['q_missing_count'] = final_df.isna().sum(axis=1)

In [ ]:
# Convert all columns to integers
final_df = final_df.astype('Int64')

### Cronbach's Alpha Validation

Before averaging the Support and Sense-of-Belonging items into composite scores, we validate internal consistency using Cronbach's alpha (α):
- α ≥ 0.9: excellent
- α ≥ 0.8: good
- α ≥ 0.7: acceptable (minimum threshold for scale aggregation)
- α < 0.7: questionable — note as a limitation and flag for future review

In [ ]:
def cronbach_alpha(df_items: pd.DataFrame) -> float:
    """
    Compute Cronbach's alpha for a set of questionnaire items.

    Uses the standard formula: α = (k / (k-1)) * (1 - Σvar_i / var_total)
    Only rows with complete data (no NaN) are used.

    Args:
        df_items: DataFrame where each column is one questionnaire item.

    Returns:
        Cronbach's alpha coefficient, or NaN if fewer than 2 complete rows.
    """
    df_clean = df_items.dropna()
    if df_clean.shape[0] < 2 or df_clean.shape[1] < 2:
        return float("nan")
    k = df_clean.shape[1]
    item_variances = df_clean.var(axis=0, ddof=1).sum()
    total_variance = df_clean.sum(axis=1).var(ddof=1)
    return (k / (k - 1)) * (1 - item_variances / total_variance)


support_cols = [col for col in final_df.columns if "Support" in col]
belonging_cols = [col for col in final_df.columns if "Sense of belonging" in col]

alpha_support = cronbach_alpha(final_df[support_cols].apply(pd.to_numeric, errors="coerce"))
alpha_belonging = cronbach_alpha(final_df[belonging_cols].apply(pd.to_numeric, errors="coerce"))

print(f"Support scale ({len(support_cols)} items): Cronbach's α = {alpha_support:.3f}")
print(f"Belonging scale ({len(belonging_cols)} items): Cronbach's α = {alpha_belonging:.3f}")

ALPHA_THRESHOLD = 0.7
for scale, alpha in [("Support", alpha_support), ("Belonging", alpha_belonging)]:
    if alpha >= ALPHA_THRESHOLD:
        print(f"✅ {scale}: α ≥ {ALPHA_THRESHOLD} — aggregation into {scale}_Avg is justified.")
    else:
        print(f"⚠️  {scale}: α < {ALPHA_THRESHOLD} — aggregation has limited reliability; noted as limitation for future review.")

In [ ]:
# Average 'Support' and 'Sense of belonging' questions into new columns
support_cols = [col for col in final_df.columns if 'Support' in col]
belonging_cols = [col for col in final_df.columns if 'Sense of belonging' in col]
final_df['Support_Avg'] = final_df[support_cols].mean(axis=1)
final_df['Belonging_Avg'] = final_df[belonging_cols].mean(axis=1)

# Drop the original 'Support' and 'Sense of belonging' columns
final_df.drop(columns=support_cols + belonging_cols, inplace=True)

With a proper analysis we need to investigate whether the support and sense of belonging questions can be averaged into one column, e.g. by calculating Cronbach's alpha to see if they measure the same underlying construct. **See Cronbach's alpha validation above** — if α ≥ 0.7 the aggregation is statistically justified; if α < 0.7, it is noted as a limitation for future work. For now we just average them into one column, and drop the original columns.

### Set up categorical variable for missingness
We bin the q_missing_count variable into three categories: 
- Non-responder (0 questions answered), 
- Partial Responder (1-11 questions answered), and 
- Complete Responder (12 or more questions answered). 

This will allow us to analyze the impact of missing data on our model.

In [ ]:
# Convert q_missing_count to categorical variable
final_df['response_type'] = pd.cut(final_df['q_missing_count'], bins=[-1, 0, 11, float('inf')], labels=['Complete-responder', 'Partial-responder', 'Non-responder'])

# Drop the q_missing_count column
final_df.drop(columns=['q_missing_count'], inplace=True)

In [ ]:
# For each of these columns, add a prefix 'sdo78'
final_df = final_df.add_prefix('sdo78_')

## Save the final dataframe to a CSV file

In [ ]:
# Save the final dataframe to a CSV file
output_path = DATA_DIR / "sdo78_combined.csv"
final_df.to_csv(output_path, index=False, encoding="utf-8-sig")

#Print where the output file is saved
print(f"Final combined dataframe saved to: {output_path}")